# 03 Rung 2: frozen XLS-R-300M, layer sweep (train -> dev)

Embeddings come from `deepvoice.xlsr.extract_cached` (mean-pooled every 3rd layer, fp16, cached in
`data/features/`). For each layer: standardized logistic regression (C=1, class-balanced, seed 0),
trained on train and scored on dev with a speaker-clustered 95% CI. Results: `results/rung2.csv`.
The layer for rung 2 is chosen on dev.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.models import run
from deepvoice.xlsr import LAYERS

p = pd.read_parquet("data/protocols.parquet")
COLS = [f"x{i:04d}" for i in range(1024)]
D = {}
for s in ["train", "dev"]:
    m = p[p.split == s][["file", "label", "speaker"]].reset_index(drop=True)
    assert Path(f"data/features/xlsr300m_{s}_files.txt").read_text().split() == list(m.file)
    D[s] = (m, np.load(f"data/features/xlsr300m_{s}.npy", mmap_mode="r"))

In [ ]:
out = ROOT / "results/rung2.csv"
done = set(pd.read_csv(out).config) if out.exists() else set()
for k, layer in enumerate(LAYERS):
    if f"rung2_logreg_xlsr300m_L{layer:02d}_s0" in done:
        continue
    fr = {s: pd.concat([m, pd.DataFrame(np.asarray(e[:, k], dtype=np.float32), columns=COLS)], axis=1)
          for s, (m, e) in D.items()}
    run(2, "logreg", COLS, fr["train"], [("asvspoof", "train", fr["train"]), ("asvspoof", "dev", fr["dev"])],
        params={"C": 1.0}, feature_set=f"xlsr300m_L{layer:02d}")
r = pd.read_csv(out)
r["layer"] = r.model.str.extract(r"L(\d+)").astype(int)[0]
dev = r[r.split == "dev"].sort_values("layer")
dev[["layer", "eer", "ci_low", "ci_high", "train_time"]].assign(**{c: dev[c] * 100 for c in ["eer", "ci_low", "ci_high"]}).round(3)

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white"})
fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.fill_between(dev.layer, 100 * dev.ci_low, 100 * dev.ci_high, color="#1baf7a", alpha=0.18, lw=0)
ax.plot(dev.layer, 100 * dev.eer, marker="o", ms=6, lw=2, color="#1baf7a", label="XLS-R layer + logistic regression")
ax.axhline(0.49, color=MUTED, ls="--", lw=1)
ax.text(24.3, 0.49, "best classical\n(LFCC40 RF, untuned)", va="center", fontsize=8, color=MUTED)
ax.set_xticks(LAYERS); ax.set_xlim(-0.5, 30)
ax.set_xlabel("XLS-R layer (0 = CNN output, 24 = last)"); ax.set_ylabel("dev EER % (band = 95% CI)")
ax.grid(axis="y", color=GRID, linewidth=0.6)
ax.set_title("Frozen XLS-R-300M: dev EER by layer", loc="left", color=INK)
fig.tight_layout()
fig.savefig(ROOT / "results/figures/xlsr_layers.png", dpi=200)
fig.savefig(ROOT / "results/figures/xlsr_layers.pdf")